In [8]:
# Load and analyze both gpt-52 simulation files
import json
import re
from collections import defaultdict

NONE_FILE = "dataset/simulation_none_dev-gpt-52-reasoning.json"
CLAR_FILE = "dataset/simulation_clarification_dev-gpt-52-reasoning.json"

with open(NONE_FILE) as f:
    none_data = json.load(f)
with open(CLAR_FILE) as f:
    clar_data = json.load(f)

none_by_idx = {e["index"]: e for e in none_data}
clar_by_idx = {e["index"]: e for e in clar_data}

def parse_index(idx):
    m = re.match(r"(query\d+)_(\d+)", idx)
    return m.group(1), int(m.group(2))

# Group entries by base query; record amb_count per variation
query_groups = defaultdict(list)
for entry in none_data:
    base, var = parse_index(entry["index"])
    query_groups[base].append({
        "index": entry["index"],
        "variation": var,
        "amb_count": len(entry["amb_specs"])
    })
for base in query_groups:
    query_groups[base].sort(key=lambda x: x["variation"])

# Print summary
print(f"Total entries: {len(none_data)}")
print(f"Unique base queries: {len(query_groups)}\n")
for base in sorted(query_groups, key=lambda x: int(x.replace("query", ""))):
    vs = query_groups[base]
    print(f"{base}: {len(vs)} variation(s), amb_counts={[v['amb_count'] for v in vs]}")

Total entries: 32
Unique base queries: 11

query0: 3 variation(s), amb_counts=[4, 4, 4]
query1: 3 variation(s), amb_counts=[1, 1, 1]
query2: 5 variation(s), amb_counts=[3, 3, 3, 3, 3]
query3: 2 variation(s), amb_counts=[1, 1]
query4: 3 variation(s), amb_counts=[1, 1, 1]
query5: 2 variation(s), amb_counts=[2, 2]
query6: 3 variation(s), amb_counts=[3, 3, 3]
query7: 4 variation(s), amb_counts=[2, 2, 2, 2]
query8: 3 variation(s), amb_counts=[4, 4, 4]
query9: 2 variation(s), amb_counts=[4, 4]
query10: 2 variation(s), amb_counts=[2, 2]


In [9]:
# Select 15 indices with balanced amb_specs distribution
from collections import Counter

# Step 1: mandatory — one entry per base query (first variation)
mandatory = {base: group[0] for base, group in query_groups.items()}

amb_dist = Counter(v["amb_count"] for v in mandatory.values())
print(f"Mandatory selection ({len(mandatory)} entries):")
print(f"  amb_specs distribution: {dict(sorted(amb_dist.items()))}")

# Step 2: pick extras from queries with >1 variation to reach 15 total and balance distribution
TARGET = 15
n_extras = TARGET - len(mandatory)

extra_candidates = []
for base, group in query_groups.items():
    for v in group[1:]:  # skip first (already in mandatory)
        extra_candidates.append(v)

# Greedy: always pick the candidate whose amb_count is currently least represented
selected_extras = []
current_dist = Counter(amb_dist)
for _ in range(n_extras):
    best = min(extra_candidates, key=lambda v: (current_dist[v["amb_count"]], v["variation"]))
    selected_extras.append(best)
    current_dist[best["amb_count"]] += 1
    extra_candidates.remove(best)

# Combine and sort by query number then variation
selected = list(mandatory.values()) + selected_extras
selected_indices = sorted(
    [s["index"] for s in selected],
    key=lambda x: (int(parse_index(x)[0].replace("query", "")), parse_index(x)[1])
)

final_dist = Counter(s["amb_count"] for s in selected)
print(f"\nFinal {len(selected_indices)} selected indices:")
for idx in selected_indices:
    entry = none_by_idx[idx]
    print(f"  {idx} (amb_specs={len(entry['amb_specs'])})")
print(f"\nFinal amb_specs distribution: {dict(sorted(final_dist.items()))}")

Mandatory selection (11 entries):
  amb_specs distribution: {1: 3, 2: 3, 3: 2, 4: 3}

Final 15 selected indices:
  query0_0 (amb_specs=4)
  query0_1 (amb_specs=4)
  query1_0 (amb_specs=1)
  query1_1 (amb_specs=1)
  query2_0 (amb_specs=3)
  query2_1 (amb_specs=3)
  query3_0 (amb_specs=1)
  query4_0 (amb_specs=1)
  query5_0 (amb_specs=2)
  query5_1 (amb_specs=2)
  query6_0 (amb_specs=3)
  query7_0 (amb_specs=2)
  query8_0 (amb_specs=4)
  query9_0 (amb_specs=4)
  query10_0 (amb_specs=2)

Final amb_specs distribution: {1: 4, 2: 4, 3: 3, 4: 4}


In [12]:
# Curate and save combined output file
def curate_entry(entry, question_id, simulation_type):
    conv_log = [
        {"content": msg["content"], "role": msg["role"]}
        for msg in entry["conv_log"][1:]
    ]
    return {
        "question_ID": question_id,
        "simulation_type": simulation_type,
        "index": entry["index"],
        "amb_specs": entry["amb_specs"],
        "conv_log": conv_log
    }

curated = []
qid = 1
for idx in selected_indices:
    curated.append(curate_entry(none_by_idx[idx], qid, "none"))
    qid += 1
    curated.append(curate_entry(clar_by_idx[idx], qid, "clarification"))
    qid += 1

with open("curated_simulation_questions.json", "w") as f:
    json.dump(curated, f, indent=2)

print(f"Saved {len(curated)} entries to curated_simulation_questions.json (IDs 1–{len(curated)})")

Saved 30 entries to curated_simulation_questions.json (IDs 1–30)


In [13]:
# Export curated questions to CSV
import csv
import json

with open("curated_simulation_questions.json") as f:
    data = json.load(f)

with open("curated_simulation_questions.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question_ID", "simulation_type", "index", "amb_specs", "conv_log"])
    writer.writeheader()
    for entry in data:
        writer.writerow({
            "question_ID": entry["question_ID"],
            "simulation_type": entry["simulation_type"],
            "index": entry["index"],
            "amb_specs": json.dumps(entry["amb_specs"]),
            "conv_log": json.dumps(entry["conv_log"])
        })

print(f"Saved {len(data)} rows to curated_simulation_questions.csv")

Saved 30 rows to curated_simulation_questions.csv
